## Logic Tests for Salescope Dashboard
This notebook verifies the core filtering and aggregation logic used in the Salescope dashboard.

It uses the refactored functions in src/logic.py & compares expected vs actual results for different filter settings.

The goal is to document expected behavior & make it clear what could break if the dashboard logic changes.

In [1]:
import sys
sys.path.append("../src")

In [2]:
import pandas as pd
from logic import normalize_range, create_summary_table, filter_sales_data

## Create a representative test dataset
This small synthetic dataset is used to verify the dashboard logic in a way that is easy to inspect manually.

In [3]:
def make_test_df():
    return pd.DataFrame({
        "Customer_ID": [1, 2, 3, 4],
        "Region": ["Asia", "Asia", "Europe", "Europe"],
        "Most_Frequent_Category": ["Clothing", "Electronics", "Clothing", "Sports"],
        "Retention_Strategy": ["Discount", "Email Campaign", "Discount", "Loyalty Program"],
        "Lifetime_Value": [500, 1000, 1500, 2000],
        "Churn_Probability": [0.10, 0.30, 0.50, 0.80],
        "Average_Order_Value": [50, 60, 70, 80],
        "Purchase_Frequency": [2, 4, 6, 8],
        "Time_Between_Purchases": [10, 20, 30, 40],
        "Launch_Date": pd.to_datetime(["2024-01-10", "2024-02-10", "2024-03-10", "2024-04-10"]),
    })

df = make_test_df()
df

,Customer_ID,Region,Most_Frequent_Category,Retention_Strategy,Lifetime_Value,Churn_Probability,Average_Order_Value,Purchase_Frequency,Time_Between_Purchases,Launch_Date
0,1,Asia,Clothing,Discount,500,0.1,50,2,10,2024-01-10
1,2,Asia,Electronics,Email Campaign,1000,0.3,60,4,20,2024-02-10
2,3,Europe,Clothing,Discount,1500,0.5,70,6,30,2024-03-10
3,4,Europe,Sports,Loyalty Program,2000,0.8,80,8,40,2024-04-10


## Test 1: Reversed numeric bounds
This checks that reversed minimum and maximum values are normalized correctly so dashboard filters still behave as expected.

In [6]:
expected = (2, 10)
actual = normalize_range(10, 2, 0, 100)

print("Expected:", expected)

print("Actual:  ", actual)

Expected: (2, 10)
Actual:   (2, 10)


## Test 2: Grouped summary table aggregation
This checks that the grouped KPI table calculations are correct for count, mean, median, maximum and total.

In [8]:
summary = create_summary_table(df, "Region", "Lifetime_Value")
summary

,Region,Count,Mean,Median,Maximum,Total
0,Asia,2,750.0,750.0,1000,1500
1,Europe,2,1750.0,1750.0,2000,3500


- Expected values for the Asia row:

    Count = 2
    Mean = 750.0
    Median = 750.0
    Maximum = 1000
    Total = 1500

In [11]:
asia_row = summary[summary["Region"] == "Asia"].iloc[0]

print("Expected Count:   2")
print("Actual Count:    ", asia_row["Count"])
print()

print("Expected Mean:    750.0")
print("Actual Mean:     ", asia_row["Mean"])
print()

print("Expected Median:  750.0")
print("Actual Median:   ", asia_row["Median"])
print()

print("Expected Maximum: 1000")
print("Actual Maximum:  ", asia_row["Maximum"])
print()

print("Expected Total:   1500")
print("Actual Total:    ", asia_row["Total"])

Expected Count:   2
Actual Count:     2

Expected Mean:    750.0
Actual Mean:      750.0

Expected Median:  750.0
Actual Median:    750.0

Expected Maximum: 1000
Actual Maximum:   1000

Expected Total:   1500
Actual Total:     1500
